# Web-Gold-40K - recovery v2.7 checkpoint selection

V2.7 is a selection-and-reproducibility correction only. It does not retrain or change the v2.6 model. It replays the completed five-epoch v2.6 report using the preregistered rule: an epoch must pass every quality gate, then eligible epochs are ranked by outcome MCC. The test split remains locked.

This notebook first uses an attached `gold_recovery_v2_6_mini_report.json` when available. Otherwise it reads the completed report embedded in the repository's executed v2.6 notebook. Attach the v2.6 Kaggle output if you also want the selected checkpoint file to be hashed; the selection replay itself does not require GPU or dataset download.

In [ ]:
# 1. Pull the exact modular implementation.
from pathlib import Path
import json, os, subprocess, sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
GIT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('git commit:', GIT_COMMIT)

In [ ]:
# 2. Prefer the original JSON report from an attached v2.6 Kaggle output.
attached_reports = sorted(Path('/kaggle/input').rglob('gold_recovery_v2_6_mini_report.json'))
if len(attached_reports) > 1:
    raise AssertionError(f'Attach exactly one v2.6 mini output; found: {attached_reports}')
if attached_reports:
    SOURCE = attached_reports[0]
    SOURCE_KIND = 'attached v2.6 JSON report'
else:
    SOURCE = REPO_ROOT / 'notebooks' / 'kaggle_gold_recovery_v2_6.ipynb'
    SOURCE_KIND = 'completed v2.6 report embedded in repository notebook'
assert SOURCE.is_file(), SOURCE
print('source kind:', SOURCE_KIND)
print('source:', SOURCE)

In [ ]:
# 3. Replay selection and export synchronized v2.7 artifacts.
REPORT_PATH = Path('/kaggle/working/gold_recovery_v2_7_selection_report.json')
CSV_PATH = Path('/kaggle/working/gold_recovery_v2_7_selection.csv')
command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'replay_v2_7_selection.py'),
    '--source', str(SOURCE),
    '--report', str(REPORT_PATH),
    '--csv', str(CSV_PATH),
    '--checkpoint-root', '/kaggle/input',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
assert REPORT_PATH.is_file() and CSV_PATH.is_file()

In [ ]:
# 4. Enforce every v2.7 consistency condition.
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
quality = report['quality_gates']
assert report['status'] == 'PASS'
assert report['test_rows_read'] == 0
assert report['selection_rule'] == 'all_gates_then_outcome_mcc'
assert quality['status'] == 'PASS'
assert quality['eligible_epochs'] == [3, 4]
assert quality['unconstrained_outcome_epoch'] == 1
assert quality['selected_epoch'] == 4
assert quality['selected_epoch_is_eligible'] is True
assert all(quality['checks'].values())
assert report['selected_epoch'] == quality['selected_epoch']
assert report['best_checkpoint'] == report['epoch_checkpoints']['4']
assert quality['selected_checkpoint'] == report['best_checkpoint']
selected = next(row for row in report['history'] if row['epoch'] == 4)
unconstrained = next(row for row in report['history'] if row['epoch'] == 1)
assert abs(report['best_metric'] - selected['outcome_mcc']) < 1e-12
assert abs(report['unconstrained_best_metric'] - unconstrained['outcome_mcc']) < 1e-12
assert report['selection_replay']['model_weights_changed'] is False
assert report['selection_replay']['selected_checkpoint_roundtrip_rerun'] is False
print('V2.7 SELECTION REPLAY PASSED')
print('eligible epochs:', quality['eligible_epochs'])
print('selected epoch:', report['selected_epoch'])
print('selected outcome MCC:', report['best_metric'])
print('selected checkpoint:', report['best_checkpoint'])
print('checkpoint artifact:', report['selected_checkpoint_artifact']['status'])
print('report:', REPORT_PATH)
print('CSV:', CSV_PATH)

## Interpretation

`PASS` means the corrected selector was replayed successfully on the unchanged v2.6 five-epoch evidence. Epoch 4 is now the synchronized selected row and checkpoint reference. `checkpoint artifact: NOT_MOUNTED` only means the prior checkpoint file was not attached to this notebook; it does not change the selection result. This replay does not claim a new training run or a new prediction round-trip. Future training should use `configs/backbones/qwen2vl_2b_gold_v2_7.yaml`, which performs the same selection before loading and round-trip testing the selected checkpoint.